In [ ]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashteamstats, leaguedashplayerstats
import time
from io import StringIO

### STEP 1: SCRAPE PLAYER POSITIONS FROM BASKETBALL-REFERENCE ###
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Print column names for debugging
    print("Columns in extracted table:", df.columns)

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    print("Checking dataframe after processing:")
    print(df.head())  # Print sample data to debug

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

### STEP 2: FETCH DEFENSIVE STATS FROM NBA API ###
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

### STEP 3: FETCH PLAYER STATS & MERGE WITH POSITIONS ###
def get_player_stats():
    player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    # Keep relevant columns
    player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
    return player_stats

### STEP 4: MERGE DATA & CALCULATE DVP ###
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)  # Avoid rate limiting

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    print("Data types after conversion:")
    print(dvp_df.dtypes)  # Check data types
    print(dvp_df.head())   # Print sample data

    # Group by Position to calculate average DvP
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    print("\n Defense vs Position (DVP) Calculated Successfully!\n")
    return dvp_summary

### RUN THE FULL PIPELINE ###
if __name__ == "__main__":
    dvp_results = calculate_dvp()
    print(dvp_results)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 5.8 MB/s eta 0:00:00
Fetching player positions...


ValueError: Table not found on Basketball-Reference. The structure may have changed.

In [ ]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats
import time
from io import StringIO

### STEP 1: SCRAPE PLAYER POSITIONS FROM BASKETBALL-REFERENCE ###
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

### STEP 2: FETCH TEAM DEFENSIVE STATS FROM NBA API ###
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

### STEP 3: FETCH PLAYER STATS FROM NBA API ###
def get_player_stats():
    player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    # Keep relevant columns
    player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
    return player_stats

### STEP 4: MERGE DATA & CALCULATE DVP ###
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

### STEP 5: PROJECT PRA FOR EACH PLAYER AGAINST OPPOSING TEAMS ###
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST']].copy()  # Create a copy here
    proj_pra_df['Proj_PRA'] = proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']

    print("Projection completed successfully!")
    return proj_pra_df


### RUN THE FULL PIPELINE ###
if __name__ == "__main__":
    projected_pra = project_pra()
    print(projected_pra.head())


Fetching player positions...
Fetching team defensive stats...
Fetching player stats...
Merging player stats with positions...
Merging with team defensive stats...
Projecting PRA...
Projection completed successfully!
     PLAYER_NAME     TEAM_ID  Pos   Proj_PTS  Proj_REB  Proj_AST   Proj_PRA
0    A.J. Lawson  1610612742   SG   3.230970  1.240446  0.513174   4.984590
1       AJ Green  1610612749  NaN   4.574995  1.109342  0.494513   6.178850
2     AJ Griffin  1610612737   SF   2.525942  0.911769  0.315742   3.753453
3   Aaron Gordon  1610612743   PF  13.306090  6.301931  3.357092  22.965114
4  Aaron Holiday  1610612745   PG   6.525525  1.646593  1.639172   9.811290


In [ ]:
import requests
import pandas as pd
import pytz
from datetime import datetime

# Function to fetch today's NBA schedule
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Example of how to call the function and get today's games
games_today = fetch_schedule()
for game in games_today:
    print(f"Game: {game['home_team']} vs {game['away_team']}")

# Now, integrate the game schedule with player projections (continue from the projection code)
def project_pra_for_todays_games():
    games_today = fetch_schedule()

    if not games_today:
        return []

    projected_pra_today = []

    # Assuming 'calculate_dvp' function is ready to use for Defense vs Position (DVP)
    # You should have functions for fetching player stats, team stats, etc. from the earlier parts of your code
    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        # Placeholder function to fetch players for each team (you need to implement this logic based on your data)
        home_players = get_players_for_team(home_team)  # You should have a function like this
        away_players = get_players_for_team(away_team)  # You should have a function like this

        # Placeholder function to calculate projections for these players
        for player in home_players:
            projected_pra = calculate_projection(player, away_team)  # Assuming you have this function
            projected_pra_today.append({
                'PLAYER_NAME': player['name'],
                'TEAM': home_team,
                'Proj_PTS': projected_pra['pts'],
                'Proj_REB': projected_pra['reb'],
                'Proj_AST': projected_pra['ast'],
                'Proj_PRA': projected_pra['pts'] + projected_pra['reb'] + projected_pra['ast']
            })

        for player in away_players:
            projected_pra = calculate_projection(player, home_team)  # Assuming you have this function
            projected_pra_today.append({
                'PLAYER_NAME': player['name'],
                'TEAM': away_team,
                'Proj_PTS': projected_pra['pts'],
                'Proj_REB': projected_pra['reb'],
                'Proj_AST': projected_pra['ast'],
                'Proj_PRA': projected_pra['pts'] + projected_pra['reb'] + projected_pra['ast']
            })

    return projected_pra_today

# Assuming you have implemented player stats fetching and projection functions
projected_pra_today = project_pra_for_todays_games()

# Print out projections for today's games
for projection in projected_pra_today:
    print(projection)



Game: BOS vs DAL
Game: MIN vs HOU
Game: DEN vs ORL
Game: LAL vs GSW
Game: POR vs SAC
Game: LAC vs IND


NameError: name 'get_players_for_team' is not defined

In [ ]:
import json
from nba_api.stats.endpoints import leaguegamefinder, leaguedashplayerstats
import pandas as pd
import datetime
import time

def get_player_positions():
    # Assuming this function already works
    pass

def get_team_defense_stats():
    # Assuming this function already works
    pass

def get_player_stats():
    player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    if 'USG_PCT' not in player_stats.columns:
        player_stats['USG_PCT'] = None  # Add a column with None for missing data

    player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST', 'USG_PCT']]
    return player_stats

def get_todays_games():
    today = datetime.date.today()
    today_str = today.strftime('%Y-%m-%d')

    print(f"Requesting games for: {today_str}")

    try:
        game_finder = leaguegamefinder.LeagueGameFinder(date_from_nullable=today_str, date_to_nullable=today_str)
        games_data = game_finder.get_json()  # Directly fetch the raw JSON response
        print("Raw response from the API:", games_data)  # Print the raw response
        games_df = pd.DataFrame(games_data['data'])  # Convert the JSON data to a DataFrame
    except Exception as e:
        print(f"Error fetching games: {e}")
        return pd.DataFrame()

    if games_df.empty:
        print(f"No games found for today ({today_str}).")
        return pd.DataFrame()

    return games_df[['GAME_ID', 'HOME_TEAM_ID', 'VISITOR_TEAM_ID']]

def project_pra_for_todays_games():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    print("Fetching today's games...")
    todays_games = get_todays_games()

    if todays_games.empty:
        print("No games for today, exiting.")
        return []

    print("Today's games:", todays_games)

    try:
        # Check the raw API response for today's games
        raw_response = game_finder.get_json()
        print("Raw response from the API:", raw_response)
    except Exception as e:
        print(f"Error while fetching raw response: {e}")

    # Continue processing as before
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left", suffixes=('_player', '_team'))
    dvp_df['TEAM_NAME'] = dvp_df['TEAM_ID'].map(team_defense_df.set_index('TEAM_ID')['TEAM_NAME'])

    todays_player_ids = pd.concat([todays_games['HOME_TEAM_ID'], todays_games['VISITOR_TEAM_ID']]).unique()
    dvp_df = dvp_df[dvp_df['TEAM_ID'].isin(todays_player_ids)]

    dvp_df["OPP_TEAM_ID"] = dvp_df["TEAM_ID"].apply(lambda x: 1610612741 if x == 1610612742 else 1610612742)
    dvp_df["OPP_PTS"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['OPP_PTS'])
    dvp_df["OPP_REB"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['OPP_REB'])
    dvp_df["OPP_AST"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['OPP_AST'])

    dvp_df["OPP_TEAM_NAME"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['TEAM_NAME'])

    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())

    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_NAME', 'Pos', 'OPP_TEAM_NAME', 'USG_PCT', 'Proj_PTS', 'Proj_REB', 'Proj_AST']].copy()
    proj_pra_df['Proj_PRA'] = proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']

    print("Projection completed successfully!")

    grouped_games = []

    for _, game in todays_games.iterrows():
        home_team = game['HOME_TEAM_ID']
        visitor_team = game['VISITOR_TEAM_ID']

        home_players = proj_pra_df[proj_pra_df['TEAM_NAME'] == team_defense_df[team_defense_df['TEAM_ID'] == home_team]['TEAM_NAME'].values[0]]
        visitor_players = proj_pra_df[proj_pra_df['TEAM_NAME'] == team_defense_df[team_defense_df['TEAM_ID'] == visitor_team]['TEAM_NAME'].values[0]]

        grouped_games.append({
            'HOME_TEAM': team_defense_df[team_defense_df['TEAM_ID'] == home_team]['TEAM_NAME'].values[0],
            'VISITOR_TEAM': team_defense_df[team_defense_df['TEAM_ID'] == visitor_team]['TEAM_NAME'].values[0],
            'HOME_PLAYERS': home_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
            'VISITOR_PLAYERS': visitor_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
        })

    return grouped_games

### RUN THE FULL PIPELINE FOR TODAY'S GAMES ###
if __name__ == "__main__":
    projected_pra_today = project_pra_for_todays_games()
    for game in projected_pra_today:
        print(f"\nGame: {game['HOME_TEAM']} vs {game['VISITOR_TEAM']}")
        print("\nHome Team Players and Projections:")
        print(game['HOME_PLAYERS'])
        print("\nVisitor Team Players and Projections:")
        print(game['VISITOR_PLAYERS'])


Fetching player positions...
Fetching team defensive stats...
Fetching player stats...
Fetching today's games...
Requesting games for: 2025-02-06
Error fetching games: Expecting value: line 1 column 1 (char 0)
No games for today, exiting.


In [ ]:
!pip install nba_api
import json
import requests  # Added this import
import pandas as pd
import pytz
from datetime import datetime
import time
from nba_api.stats.endpoints import leaguegamefinder, leaguedashplayerstats

# Function to fetch player positions
def get_player_positions():
    # Assuming this function already works or is defined elsewhere
    pass

# Function to fetch team defense stats
def get_team_defense_stats():
    # Assuming this function already works or is defined elsewhere
    pass

# Function to fetch player stats
def get_player_stats():
    player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    if 'USG_PCT' not in player_stats.columns:
        player_stats['USG_PCT'] = None  # Add a column with None for missing data

    player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST', 'USG_PCT']]
    return player_stats

# Function to fetch today's NBA schedule
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Main projection function
def project_pra_for_todays_games():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    print("Fetching today's games...")
    todays_games = fetch_schedule()  # Corrected to use fetch_schedule

    if not todays_games:
        print("No games for today, exiting.")
        return []

    print("Today's games:", todays_games)

    # Merging player stats with player positions
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merging with team defense stats
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left", suffixes=('_player', '_team'))
    dvp_df['TEAM_NAME'] = dvp_df['TEAM_ID'].map(team_defense_df.set_index('TEAM_ID')['TEAM_NAME'])

    todays_player_ids = pd.concat([pd.Series([game['home_team_id'] for game in todays_games]),
                                   pd.Series([game['away_team_id'] for game in todays_games])]).unique()

    dvp_df = dvp_df[dvp_df['TEAM_ID'].isin(todays_player_ids)]

    # Add opponent team stats
    dvp_df["OPP_TEAM_ID"] = dvp_df["TEAM_ID"].apply(lambda x: 1610612741 if x == 1610612742 else 1610612742)
    dvp_df["OPP_PTS"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['OPP_PTS'])
    dvp_df["OPP_REB"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['OPP_REB'])
    dvp_df["OPP_AST"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['OPP_AST'])
    dvp_df["OPP_TEAM_NAME"] = dvp_df["OPP_TEAM_ID"].map(team_defense_df.set_index('TEAM_ID')['TEAM_NAME'])

    # Convert numeric columns to numeric
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Projection calculations
    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())

    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_NAME', 'Pos', 'OPP_TEAM_NAME', 'USG_PCT', 'Proj_PTS', 'Proj_REB', 'Proj_AST']].copy()
    proj_pra_df['Proj_PRA'] = proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']

    print("Projection completed successfully!")

    grouped_games = []

    # Grouping players based on teams and game
    for game in todays_games:
        home_team = game['home_team_id']
        visitor_team = game['away_team_id']

        home_players = proj_pra_df[proj_pra_df['TEAM_NAME'] == team_defense_df[team_defense_df['TEAM_ID'] == home_team]['TEAM_NAME'].values[0]]
        visitor_players = proj_pra_df[proj_pra_df['TEAM_NAME'] == team_defense_df[team_defense_df['TEAM_ID'] == visitor_team]['TEAM_NAME'].values[0]]

        grouped_games.append({
            'HOME_TEAM': team_defense_df[team_defense_df['TEAM_ID'] == home_team]['TEAM_NAME'].values[0],
            'VISITOR_TEAM': team_defense_df[team_defense_df['TEAM_ID'] == visitor_team]['TEAM_NAME'].values[0],
            'HOME_PLAYERS': home_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
            'VISITOR_PLAYERS': visitor_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
        })

    return grouped_games

### RUN THE FULL PIPELINE FOR TODAY'S GAMES ###
if __name__ == "__main__":
    projected_pra_today = project_pra_for_todays_games()
    for game in projected_pra_today:
        print(f"\nGame: {game['HOME_TEAM']} vs {game['VISITOR_TEAM']}")
        print("\nHome Team Players and Projections:")
        print(game['HOME_PLAYERS'])
        print("\nVisitor Team Players and Projections:")
        print(game['VISITOR_PLAYERS'])


Fetching player positions...
Fetching team defensive stats...
Fetching player stats...
Fetching today's games...
Today's games: [{'home_team': 'BOS', 'away_team': 'DAL', 'home_team_id': 1610612738, 'away_team_id': 1610612742}, {'home_team': 'MIN', 'away_team': 'HOU', 'home_team_id': 1610612750, 'away_team_id': 1610612745}, {'home_team': 'DEN', 'away_team': 'ORL', 'home_team_id': 1610612743, 'away_team_id': 1610612753}, {'home_team': 'LAL', 'away_team': 'GSW', 'home_team_id': 1610612747, 'away_team_id': 1610612744}, {'home_team': 'POR', 'away_team': 'SAC', 'home_team_id': 1610612757, 'away_team_id': 1610612758}, {'home_team': 'LAC', 'away_team': 'IND', 'home_team_id': 1610612746, 'away_team_id': 1610612754}]


TypeError: Can only merge Series or DataFrame objects, a <class 'NoneType'> was passed

In [ ]:
import pandas as pd
import requests
import pytz
from datetime import datetime
import time

# Mocked functions for demonstration purposes
def get_player_positions():
    # Assuming this returns a valid DataFrame with player positions
    return pd.DataFrame({
        'PLAYER_NAME': ['Player1', 'Player2'],
        'Pos': ['PG', 'SG'],
        'TEAM_ID': [1610612738, 1610612742]
    })

def get_team_defense_stats():
    # Assuming this returns a valid DataFrame with team defense stats
    return pd.DataFrame({
        'TEAM_ID': [1610612738, 1610612742],
        'TEAM_NAME': ['Boston Celtics', 'Dallas Mavericks'],
        'OPP_PTS': [110, 105],
        'OPP_REB': [45, 42],
        'OPP_AST': [24, 22]
    })

# Function to fetch today's NBA schedule
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return pd.DataFrame(games_today)
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return pd.DataFrame()

def get_player_stats():
    # Mocked player stats (you can replace this with actual NBA data fetch)
    return pd.DataFrame({
        'PLAYER_ID': [1, 2],
        'PLAYER_NAME': ['Player1', 'Player2'],
        'TEAM_ID': [1610612738, 1610612742],
        'PTS': [25, 22],
        'REB': [5, 4],
        'AST': [7, 5],
        'USG_PCT': [20, 22]
    })

def project_pra_for_todays_games():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    print("Team Defense Data:", team_defense_df)  # Debugging line to see team defense data
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    print("Fetching today's games...")
    todays_games = fetch_schedule()

    if todays_games.empty:
        print("No games for today, exiting.")
        return []

    print("Today's games:", todays_games)

    # Merge player stats with positions (renaming conflicting columns)
    merged_df = player_stats_df.merge(player_positions_df, on="PLAYER_NAME", how="left")
    merged_df.rename(columns={"TEAM_ID_x": "TEAM_ID"}, inplace=True)  # Keep 'TEAM_ID' consistent
    print("Merged player stats with positions:\n", merged_df)

    # Add a PRA column for projections (Ensure these columns are created)
    merged_df['Proj_PTS'] = merged_df['PTS']
    merged_df['Proj_REB'] = merged_df['REB']
    merged_df['Proj_AST'] = merged_df['AST']
    merged_df['Proj_PRA'] = merged_df['Proj_PTS'] + merged_df['Proj_REB'] + merged_df['Proj_AST']

    # Check if the new columns exist
    print("Columns after projection:", merged_df.columns)

    # Merge with team defense stats
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")
    print("Merged with team defense stats:\n", dvp_df)

    dvp_df['TEAM_NAME'] = dvp_df['TEAM_ID'].map(team_defense_df.set_index('TEAM_ID')['TEAM_NAME'])

    # Debugging: Check for any missing teams
    missing_teams = set(todays_games['home_team_id']).union(set(todays_games['away_team_id'])) - set(dvp_df['TEAM_ID'])
    if missing_teams:
        print(f"Missing teams for IDs: {missing_teams}")

    # Filter out games with missing teams
    valid_games = todays_games[todays_games['home_team_id'].isin(dvp_df['TEAM_ID']) &
                                todays_games['away_team_id'].isin(dvp_df['TEAM_ID'])]

    grouped_games = []

    for _, game in valid_games.iterrows():
        home_team = game['home_team_id']
        visitor_team = game['away_team_id']

        home_team_name = team_defense_df[team_defense_df['TEAM_ID'] == home_team]
        if home_team_name.empty:
            print(f"Error: No team found for home team ID {home_team}")
            continue

        visitor_team_name = team_defense_df[team_defense_df['TEAM_ID'] == visitor_team]
        if visitor_team_name.empty:
            print(f"Error: No team found for visitor team ID {visitor_team}")
            continue

        home_players = dvp_df[dvp_df['TEAM_NAME'] == home_team_name['TEAM_NAME'].values[0]]
        visitor_players = dvp_df[dvp_df['TEAM_NAME'] == visitor_team_name['TEAM_NAME'].values[0]]

        grouped_games.append({
            'HOME_TEAM': home_team_name['TEAM_NAME'].values[0],
            'VISITOR_TEAM': visitor_team_name['TEAM_NAME'].values[0],
            'HOME_PLAYERS': home_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
            'VISITOR_PLAYERS': visitor_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
        })

    return grouped_games

    for _, game in valid_games.iterrows():
        home_team = game['home_team_id']
        visitor_team = game['away_team_id']

        home_team_name = team_defense_df[team_defense_df['TEAM_ID'] == home_team]
        if home_team_name.empty:
            print(f"Error: No team found for home team ID {home_team}")
            continue

        visitor_team_name = team_defense_df[team_defense_df['TEAM_ID'] == visitor_team]
        if visitor_team_name.empty:
            print(f"Error: No team found for visitor team ID {visitor_team}")
            continue

        home_players = dvp_df[dvp_df['TEAM_NAME'] == home_team_name['TEAM_NAME'].values[0]]
        visitor_players = dvp_df[dvp_df['TEAM_NAME'] == visitor_team_name['TEAM_NAME'].values[0]]

        grouped_games.append({
            'HOME_TEAM': home_team_name['TEAM_NAME'].values[0],
            'VISITOR_TEAM': visitor_team_name['TEAM_NAME'].values[0],
            'HOME_PLAYERS': home_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
            'VISITOR_PLAYERS': visitor_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']],
        })

    return grouped_games



### RUN THE FULL PIPELINE FOR TODAY'S GAMES ###
if __name__ == "__main__":
    projected_pra_today = project_pra_for_todays_games()
    for game in projected_pra_today:
        print(f"\nGame: {game['HOME_TEAM']} vs {game['VISITOR_TEAM']}")
        print("\nHome Team Players and Projections:")
        print(game['HOME_PLAYERS'])
        print("\nVisitor Team Players and Projections:")
        print(game['VISITOR_PLAYERS'])



Fetching player positions...
Fetching team defensive stats...
Team Defense Data:       TEAM_ID         TEAM_NAME  OPP_PTS  OPP_REB  OPP_AST
0  1610612738    Boston Celtics      110       45       24
1  1610612742  Dallas Mavericks      105       42       22
Fetching player stats...
Fetching today's games...
Today's games:   home_team away_team  home_team_id  away_team_id
0       BOS       DAL    1610612738    1610612742
1       MIN       HOU    1610612750    1610612745
2       DEN       ORL    1610612743    1610612753
3       LAL       GSW    1610612747    1610612744
4       POR       SAC    1610612757    1610612758
5       LAC       IND    1610612746    1610612754
Merged player stats with positions:
    PLAYER_ID PLAYER_NAME     TEAM_ID  PTS  REB  AST  USG_PCT Pos   TEAM_ID_y
0          1     Player1  1610612738   25    5    7       20  PG  1610612738
1          2     Player2  1610612742   22    4    5       22  SG  1610612742
Columns after projection: Index(['PLAYER_ID', 'PLAYER_NAME

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats
import time
from io import StringIO
from datetime import datetime
import pytz

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2025_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Step 3: Fetch player stats from NBA API
def get_player_stats():
    player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    # Keep relevant columns
    player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
    return player_stats

# Step 4: Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Step 5: Merge data & calculate DVP
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Step 6: Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean()).round(1)
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean()).round(1)
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean()).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST']].copy()  # Create a copy here
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Print projections for today's game
        print(f"\nGame: {game['home_team']} vs {game['away_team']}")
        print(f"Projected PRA for Home Team ({game['home_team']}):")
        print(home_team_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']])
        print(f"Projected PRA for Away Team ({game['away_team']}):")
        print(away_team_players[['PLAYER_NAME', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']])

if __name__ == "__main__":
    run_pipeline_for_todays_games()


Fetching today's schedule...
Projecting PRA for players...
Fetching player positions...
Fetching team defensive stats...
Fetching player stats...
Merging player stats with positions...
Merging with team defensive stats...
Projecting PRA...
Projection completed successfully!

Game: BOS vs DAL
Projected PRA for Home Team (BOS):
            PLAYER_NAME  Proj_PTS  Proj_REB  Proj_AST  Proj_PRA
10           Al Horford       8.6       6.4      2.34      17.3
148       Derrick White      15.2       4.2      4.68      24.1
168       Drew Peterson       3.7       0.3      0.27       4.3
224          JD Davison       2.0       1.3      1.17       4.5
236      Jaden Springer       3.3       1.6      0.81       5.7
275        Jaylen Brown      23.0       5.5      3.24      31.7
278        Jayson Tatum      26.9       8.1      4.41      39.4
310        Jordan Walsh       1.7       2.2      0.54       4.4
321        Jrue Holiday      12.5       5.4      4.32      22.2
366  Kristaps Porziņģis      20.

In [ ]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics
import time
from io import StringIO
from datetime import datetime
import pytz

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        # Select necessary columns
        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Step 3: Fetch player stats (use LeaguedashPlayerStats for current season data)
def fetch_player_stats():
    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season="2024-25", per_mode_detailed="PerGame"
        ).get_data_frames()[0]

        player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
        return player_stats
    except Exception as e:
        print(f"Error fetching player stats: {e}")
        return pd.DataFrame()

# Step 4: Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Step 5: Merge data & calculate DVP with E_USG_PCT
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation, including 'E_USG_PCT'
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Step 6: Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()  # Fetch the USG% data
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Merge with player usage percentage (E_USG_PCT)
    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean()).round(1)
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean()).round(1)
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean()).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()  # Add USG_PCT
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    # Sort by 'E_USG_PCT' in descending order (highest usage first)
    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
        'LAL': 'Los Angeles Lakers',
        'BOS': 'Boston Celtics',
        'GSW': 'Golden State Warriors',
        'MIA': 'Miami Heat',
        'NYK': 'New York Knicks',
        'LAC': 'Los Angeles Clippers',
        # Add other teams here...
    }

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame: {home_team_full} vs {away_team_full}")
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats to ensure no wrapping
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

if __name__ == "__main__":
    run_pipeline_for_todays_games()



from google.colab import data_table
data_table.disable_dataframe_formatter()

Fetching today's schedule...
Projecting PRA for players...
Fetching player positions...
Fetching team defensive stats...
Fetching player stats...
Fetching player usage percentage...
Merging player stats with positions...
Merging with team defensive stats...
Projecting PRA...
Projection completed successfully!

Game: Boston Celtics vs DAL

Projected PRA for Home Team (Boston Celtics):
Jayson Tatum              | PTS: 26.8  | REB: 8.8   | AST: 5.1   | PRA: 40.7  | USG: 0.30 
Jaylen Brown              | PTS: 23.1  | REB: 6.1   | AST: 4.3   | PRA: 33.5  | USG: 0.29 
Kristaps Porziņģis        | PTS: 19.2  | REB: 7.1   | AST: 1.9   | PRA: 28.2  | USG: 0.25 
JD Davison                | PTS: 1.2   | REB: 0.4   | AST: 0.2   | PRA: 1.8   | USG: 0.22 
Derrick White             | PTS: 16.2  | REB: 4.2   | AST: 3.8   | PRA: 24.2  | USG: 0.19 
Payton Pritchard          | PTS: 13.7  | REB: 3.6   | AST: 3.1   | PRA: 20.4  | USG: 0.18 
Jrue Holiday              | PTS: 11.0  | REB: 4.4   | AST: 3.3   | 

In [ ]:
import pandas as pd
import requests
from nba_api.stats.endpoints import leaguedashteamstats, playerestimatedmetrics
import time
from datetime import datetime
import pytz

# Step 1: Fetch player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Rename team column to match the NBA API column
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        # Return the data as a DataFrame (not a dictionary)
        return usg_data
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of an error

# Step 3: Merge player stats with positions
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_usg()
    time.sleep(2)

    # Ensure 'TEAM_ID' is in player stats and defense data for merging
    print("Ensuring TEAM_ID is in player stats and team defense...")
    player_stats_df['TEAM_ID'] = player_stats_df['PLAYER_NAME'].apply(lambda x: get_team_id_from_player(x))  # Adjust logic for team ID
    team_defense_df['TEAM_ID'] = team_defense_df['TEAM_NAME'].apply(lambda x: get_team_id_from_team_name(x))  # Adjust for team names

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean()).round(1)
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean()).round(1)
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean()).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'USG_PCT', 'Proj_PTS', 'Proj_REB', 'Proj_AST']].copy()  # Add USG_PCT
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    # Set the index to 'USG_PCT' for sorting
    proj_pra_df.set_index('USG_PCT', inplace=True)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 4: Project PRA
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_usg()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean()).round(1)
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean()).round(1)
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean()).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'USG_PCT', 'Proj_PTS', 'Proj_REB', 'Proj_AST']].copy()  # Add USG_PCT
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    # Set the index to 'USG_PCT' for sorting
    proj_pra_df.set_index('USG_PCT', inplace=True)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 5: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Calculate and display DVP per position
    print("\nDefensive Stats vs Each Position (DVP):")
    dvp_summary = calculate_dvp()
    print(dvp_summary)

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Print projections for today's game
        print(f"\nGame: {game['home_team']} vs {game['away_team']}")
        print(f"Projected PRA for Home Team ({game['home_team']}):")
        print(home_team_players[['PLAYER_NAME', 'USG_PCT', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']])
        print(f"Projected PRA for Away Team ({game['away_team']}):")
        print(away_team_players[['PLAYER_NAME', 'USG_PCT', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'Proj_PRA']])

if __name__ == "__main__":
    run_pipeline_for_todays_games()


Fetching today's schedule...
Projecting PRA for players...
Fetching player positions...
Fetching team defensive stats...
Fetching player stats...
Merging player stats with positions...
Merging with team defensive stats...


KeyError: 'TEAM_ID'

In [1]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats
import time
from io import StringIO

### STEP 1: SCRAPE PLAYER POSITIONS FROM BASKETBALL-REFERENCE ###
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2025_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

### STEP 2: FETCH TEAM DEFENSIVE STATS FROM NBA API ###
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

### STEP 3: FETCH PLAYER STATS FROM NBA API ###
def get_player_stats():
    player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
        per_mode_detailed="PerGame",
        season="2023-24"
    ).get_data_frames()[0]

    # Keep relevant columns
    player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
    return player_stats

### STEP 4: MERGE DATA & CALCULATE DVP ###
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

### STEP 5: PROJECT PRA FOR EACH PLAYER AGAINST OPPOSING TEAMS ###
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = get_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())
    dvp_df["Proj_REB"] = dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())
    dvp_df["Proj_AST"] = dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST']].copy()  # Create a copy here
    proj_pra_df['Proj_PRA'] = proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']

    print("Projection completed successfully!")
    return proj_pra_df

### RUN THE FULL PIPELINE ###
if __name__ == "__main__":
    projected_pra = project_pra()
    print(projected_pra.head())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 8.5 MB/s eta 0:00:00
Fetching player positions...


ValueError: Table not found on Basketball-Reference. The structure may have changed.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics
import time
from io import StringIO
from datetime import datetime
import pytz

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        # Select necessary columns
        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Step 3: Fetch player stats (use LeaguedashPlayerStats for current season data)
def fetch_player_stats():
    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season="2024-25", per_mode_detailed="PerGame"
        ).get_data_frames()[0]

        player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
        return player_stats
    except Exception as e:
        print(f"Error fetching player stats: {e}")
        return pd.DataFrame()

# Step 4: Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Step 5: Merge data & calculate DVP with E_USG_PCT
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation, including 'E_USG_PCT'
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Step 6: Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()  # Fetch the USG% data
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Merge with player usage percentage (E_USG_PCT)
    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = (dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())).round(1)
    dvp_df["Proj_REB"] = (dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())).round(1)
    dvp_df["Proj_AST"] = (dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()  # Add USG_PCT
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    # Sort by 'E_USG_PCT' in descending order (highest usage first)
    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame: {home_team_full} vs {away_team_full}")
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

if __name__ == "__main__":
    run_pipeline_for_todays_games()


Fetching today's schedule...
Projecting PRA for players...
Fetching player positions...
Fetching team defensive stats...
Fetching player stats...
Fetching player usage percentage...
Merging player stats with positions...
Merging with team defensive stats...
Projecting PRA...
Projection completed successfully!

Game: Boston Celtics vs DAL

Projected PRA for Home Team (Boston Celtics):
Jayson Tatum              | PTS: 25.7  | REB: 8.9   | AST: 5.2   | PRA: 39.8  | USG: 0.30 
Jaylen Brown              | PTS: 22.1  | REB: 6.2   | AST: 4.3   | PRA: 32.6  | USG: 0.29 
Kristaps Porziņģis        | PTS: 18.4  | REB: 7.2   | AST: 1.9   | PRA: 27.5  | USG: 0.25 
JD Davison                | PTS: 1.1   | REB: 0.4   | AST: 0.2   | PRA: 1.7   | USG: 0.22 
Derrick White             | PTS: 15.5  | REB: 4.3   | AST: 3.8   | PRA: 23.6  | USG: 0.19 
Payton Pritchard          | PTS: 13.1  | REB: 3.7   | AST: 3.1   | PRA: 19.9  | USG: 0.18 
Jrue Holiday              | PTS: 10.5  | REB: 4.5   | AST: 3.3   | 

In [ ]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics
import time
from io import StringIO
from datetime import datetime
import pytz

# Assuming your DataFrame is called 'df'
df_cleaned = df.dropna()  # Removes rows with any NaN

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        # Select necessary columns
        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Step 3: Fetch player stats (use LeaguedashPlayerStats for current season data)
def fetch_player_stats():
    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season="2024-25", per_mode_detailed="PerGame"
        ).get_data_frames()[0]

        player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
        return player_stats
    except Exception as e:
        print(f"Error fetching player stats: {e}")
        return pd.DataFrame()

# Step 4: Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Step 5: Merge data & calculate DVP with E_USG_PCT
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation, including 'E_USG_PCT'
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Step 6: Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()  # Fetch the USG% data
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Merge with player usage percentage (E_USG_PCT)
    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = (dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())).round(1)
    dvp_df["Proj_REB"] = (dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())).round(1)
    dvp_df["Proj_AST"] = (dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()  # Add USG_PCT
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    # Sort by 'E_USG_PCT' in descending order (highest usage first)
    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame: {home_team_full} vs {away_team_full}")
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

# Fetch NBA injury data
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    # Print player name and status for each injury
    for team in data.get('injuries', []):
        for injury in team.get('injuries', []):
            player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
            status = injury.get('status', 'Unknown Status')
            print(f"Player: {player_name}, Status: {status}")
else:
    print("Failed to retrieve injury data.")

if __name__ == "__main__":
    run_pipeline_for_todays_games()


NameError: name 'df' is not defined

In [ ]:
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()
    time.sleep(2)

    print("Fetching injuries...")
    injuries = fetch_injuries()
    time.sleep(2)

    if player_stats_df is None or team_defense_df is None or player_positions_df is None:
        print("Error: One or more dataframes are empty. Returning empty DataFrame.")
        return pd.DataFrame()

    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = (dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())).round(1)
    dvp_df["Proj_REB"] = (dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())).round(1)
    dvp_df["Proj_AST"] = (dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())).round(1)

    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    proj_pra_df = merge_with_injuries(proj_pra_df, injuries)

    print("Projection completed successfully!")

    return proj_pra_df if not proj_pra_df.empty else pd.DataFrame()


In [ ]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics
import time
from io import StringIO
from datetime import datetime
import pytz

# Fetch NBA injury data
def fetch_injuries():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    response = requests.get(url)
    injuries = {}

    if response.status_code == 200:
        data = response.json()
        # Create a dictionary of player injuries
        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
                status = injury.get('status', 'Unknown Status')
                injuries[player_name] = status
    else:
        print("Failed to retrieve injury data.")
    return injuries

# Fetch player positions, team defense, player stats, etc. (keep these functions as is)

# Merge the injury status with player data and ensure no duplicates
def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df['Injury_Status'] = proj_pra_df['Injury_Status'].fillna('Healthy')
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df

# Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()
    time.sleep(2)

    print("Fetching injuries...")
    injuries = fetch_injuries()
    time.sleep(2)

    if player_stats_df is None or team_defense_df is None or player_positions_df is None:
        print("Error: One or more dataframes are empty. Returning empty DataFrame.")
        return pd.DataFrame()

    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = (dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())).round(1)
    dvp_df["Proj_REB"] = (dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())).round(1)
    dvp_df["Proj_AST"] = (dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())).round(1)

    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    proj_pra_df = merge_with_injuries(proj_pra_df, injuries)

    print("Projection completed successfully!")

    return proj_pra_df if not proj_pra_df.empty else pd.DataFrame()

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame: {home_team_full} vs {away_team_full}")
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

# Run the pipeline
if __name__ == "__main__":
    run_pipeline_for_todays_games()


Fetching today's schedule...


NameError: name 'fetch_schedule' is not defined

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics
import time
from io import StringIO
from datetime import datetime
import pytz

# Fetch NBA injury data
def fetch_injuries():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    response = requests.get(url)
    injuries = {}

    if response.status_code == 200:
        data = response.json()
        # Create a dictionary of player injuries
        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
                status = injury.get('status', 'Unknown Status')
                injuries[player_name] = status
    else:
        print("Failed to retrieve injury data.")
    return injuries

# Fetch player positions, team defense, player stats, etc. (keep these functions as is)

# Merge the injury status with player data and ensure no duplicates
def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df

# Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()  # Fetch the USG% data
    time.sleep(2)

    print("Fetching injuries...")
    injuries = fetch_injuries()  # Fetch injury data
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Merge with player usage percentage (E_USG_PCT)
    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = (dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())).round(1)
    dvp_df["Proj_REB"] = (dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())).round(1)
    dvp_df["Proj_AST"] = (dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()  # Add USG_PCT
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    # Sort by 'E_USG_PCT' in descending order (highest usage first)
    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    # Merge with injury status
    proj_pra_df = merge_with_injuries(proj_pra_df, injuries)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame: {home_team_full} vs {away_team_full}")
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

# Run the pipeline
if __name__ == "__main__":
    run_pipeline_for_todays_games()


Fetching today's schedule...


NameError: name 'fetch_schedule' is not defined

In [ ]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics
import time
from io import StringIO
from datetime import datetime
import pytz

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        # Select necessary columns
        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Step 3: Fetch player stats (use LeaguedashPlayerStats for current season data)
def fetch_player_stats():
    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season="2024-25", per_mode_detailed="PerGame"
        ).get_data_frames()[0]

        player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
        return player_stats
    except Exception as e:
        print(f"Error fetching player stats: {e}")
        return pd.DataFrame()

# Step 4: Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Step 5: Merge data & calculate DVP with E_USG_PCT
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation, including 'E_USG_PCT'
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Step 6: Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()  # Fetch the USG% data
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Merge with player usage percentage (E_USG_PCT)
    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    # Convert numeric columns before aggregation
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    # Project PRA for each player
    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = (dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())).round(1)
    dvp_df["Proj_REB"] = (dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())).round(1)
    dvp_df["Proj_AST"] = (dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())).round(1)

    # Clean up and return the projection
    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()  # Add USG_PCT
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    # Sort by 'E_USG_PCT' in descending order (highest usage first)
    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    print("Projection completed successfully!")
    return proj_pra_df

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame: {home_team_full} vs {away_team_full}")
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f}")

# Fetch NBA injury data
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    # Print player name and status for each injury
    for team in data.get('injuries', []):
        for injury in team.get('injuries', []):
            player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
            status = injury.get('status', 'Unknown Status')
            print(f"Player: {player_name}, Status: {status}")
else:
    print("Failed to retrieve injury data.")

if __name__ == "__main__":
    run_pipeline_for_todays_games()


Player: Clint Capela, Status: Out
Player: Larry Nance Jr., Status: Out
Player: Vit Krejci, Status: Out
Player: Jalen Johnson, Status: Out
Player: Kobe Bufkin, Status: Out
Player: Jaylen Brown, Status: Out
Player: Xavier Tillman, Status: Out
Player: Jrue Holiday, Status: Out
Player: Bojan Bogdanovic, Status: Out
Player: Noah Clowney, Status: Out
Player: Cam Thomas, Status: Out
Player: Moussa Diabate, Status: Day-To-Day
Player: E.J. Liddell, Status: Day-To-Day
Player: Patrick Williams, Status: Out
Player: Lonzo Ball, Status: Out
Player: Adama Sanogo, Status: Out
Player: Ty Jerome, Status: Out
Player: JT Thor, Status: Out
Player: Isaac Okoro, Status: Out
Player: Dean Wade, Status: Out
Player: Daniel Gafford, Status: Out
Player: Klay Thompson, Status: Out
Player: Kyrie Irving, Status: Out
Player: P.J. Washington, Status: Out
Player: Dwight Powell, Status: Out
Player: Anthony Davis, Status: Out
Player: Caleb Martin, Status: Out
Player: Dereck Lively II, Status: Out
Player: Michael Porter Jr

In [ ]:
!pip install nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics,teamestimatedmetrics
import time
from io import StringIO
from datetime import datetime
import pytz
import json
from scipy.stats import zscore

pd.set_option('display.max_columns', None)

# Team name mappings
team_name_mapping = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS",
}
team_abbreviation_to_name = {abbr: full_name for full_name, abbr in team_name_mapping.items()}

def fetch_schedule():
    """Fetch today's NBA schedule."""
    url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
    response = requests.get(url)
    data = response.json()

    est = pytz.timezone('US/Eastern')
    today = datetime.now(est).strftime('%Y-%m-%d')

    return [
        {'home_team': game['h']['ta'], 'away_team': game['v']['ta']}
        for month in data['lscd']
        for game in month['mscd']['g']
        if game['gdte'] == today
    ]

def fetch_team_metrics():
    """Fetch current team efficiency metrics."""
    team_metrics_data = teamestimatedmetrics.TeamEstimatedMetrics(season="2024-25", season_type="Regular Season")
    team_metrics = team_metrics_data.get_data_frames()[0]
    team_metrics = team_metrics[["TEAM_NAME", "E_OFF_RATING", "E_DEF_RATING", "E_PACE"]]
    team_metrics["TEAM_ABBREVIATION"] = team_metrics["TEAM_NAME"].map(team_name_mapping)

    return team_metrics

def calculate_thresholds(team_metrics):
    """Calculate thresholds for OffRtg, DefRtg, and Pace."""
    thresholds = {}

    for stat in ["E_OFF_RATING", "E_DEF_RATING", "E_PACE"]:
        values = team_metrics[stat]

        # Percentile-based (Quartiles)
        high_percentile = values.quantile(0.75)
        low_percentile = values.quantile(0.25)

        # Z-score method (above/below 1 std deviation)
        z_scores = zscore(values)
        high_zscore = values[z_scores > 1].mean() if any(z_scores > 1) else high_percentile
        low_zscore = values[z_scores < -1].mean() if any(z_scores < -1) else low_percentile

        # Absolute thresholds
        absolute_high = {"E_OFF_RATING": 115, "E_DEF_RATING": 105, "E_PACE": 100}
        absolute_low = {"E_OFF_RATING": 108, "E_DEF_RATING": 112, "E_PACE": 96}

        thresholds[stat] = {
            "high_percentile": high_percentile, "low_percentile": low_percentile,
            "high_zscore": high_zscore, "low_zscore": low_zscore,
            "absolute_high": absolute_high[stat], "absolute_low": absolute_low[stat]
        }

    return thresholds

def predict_winner_and_props(home_team, away_team, team_metrics, thresholds, hca=2.5):
    """Predict game winner using ORTG, DRTG, and Pace, and determine best prop bets."""
    home_stats = team_metrics[team_metrics["TEAM_ABBREVIATION"] == home_team].iloc[0]
    away_stats = team_metrics[team_metrics["TEAM_ABBREVIATION"] == away_team].iloc[0]

    home_team_full = team_abbreviation_to_name.get(home_team, home_team)
    away_team_full = team_abbreviation_to_name.get(away_team, away_team)

    # Extract team metrics
    home_ortg, home_drtg, home_pace = home_stats["E_OFF_RATING"], home_stats["E_DEF_RATING"], home_stats["E_PACE"]
    away_ortg, away_drtg, away_pace = away_stats["E_OFF_RATING"], away_stats["E_DEF_RATING"], away_stats["E_PACE"]

    # Calculate expected scores
    avg_pace = (home_pace + away_pace) / 2
    home_expected_score = ((home_ortg + away_drtg) / 2) * (avg_pace / 100)
    away_expected_score = ((away_ortg + home_drtg) / 2) * (avg_pace / 100)

    # Adjust for home-court advantage
    home_expected_score += hca

    # Calculate expected margin
    expected_margin = home_expected_score - away_expected_score

    # Predict winner
    winner = home_team_full if expected_margin > 0 else away_team_full

    # Determine prop bet recommendations
    avg_ortg = (home_ortg + away_ortg) / 2
    avg_drtg = (home_drtg + away_drtg) / 2

    ortg_threshold = thresholds["E_OFF_RATING"]
    drtg_threshold = thresholds["E_DEF_RATING"]
    pace_threshold = thresholds["E_PACE"]

    if avg_ortg >= ortg_threshold["high_percentile"] and avg_pace >= pace_threshold["high_percentile"]:
        prop_bets = "High-scoring game (Favoring Points & Assists Props)"
    elif avg_drtg < drtg_threshold["low_zscore"] and avg_pace <= pace_threshold["low_zscore"]:
        prop_bets = "Slow-paced game (Favoring Unders for Points Props)"
    elif avg_ortg >= ortg_threshold["absolute_high"] and avg_drtg <= drtg_threshold["absolute_low"]:
        prop_bets = "Fast-paced game (High Points Props)"
    elif avg_pace >= pace_threshold["absolute_high"]:
        prop_bets = "Fast-paced game (Big Upside for Points, Rebounds & Assists Props)"
    else:
        prop_bets = "Neutral Matchup"

    return {
        "Home Team": home_team_full,
        "Home Stats": f"OffRtg: {home_ortg} | DefRtg: {home_drtg} | Pace: {home_pace}",
        "Away Team": away_team_full,
        "Away Stats": f"OffRtg: {away_ortg} | DefRtg: {away_drtg} | Pace: {away_pace}",
        "Predicted Winner": winner,
        "Expected Margin": round(abs(expected_margin), 1),
        "Prop Bets": prop_bets
    }

# Run the Analysis
schedule = fetch_schedule()
team_metrics = fetch_team_metrics()

if not team_metrics.empty:
    thresholds = calculate_thresholds(team_metrics)
    matchups = [predict_winner_and_props(game["home_team"], game["away_team"], team_metrics, thresholds) for game in schedule]
    print(json.dumps(matchups, indent=4))

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        # Select necessary columns
        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Step 3: Fetch player stats (use LeaguedashPlayerStats for current season data)
def fetch_player_stats():
    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season="2024-25", per_mode_detailed="PerGame"
        ).get_data_frames()[0]

        player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
        return player_stats
    except Exception as e:
        print(f"Error fetching player stats: {e}")
        return pd.DataFrame()

# Step 4: Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Step 5: Merge data & calculate DVP with E_USG_PCT
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation, including 'E_USG_PCT'
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Fetch NBA injury data and store it in a dictionary
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    # Dictionary to store player injury statuses
    players_injuries = {}

    for team in data.get('injuries', []):
        for injury in team.get('injuries', []):
            player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
            status = injury.get('status', '').strip()  # Get status and remove whitespace

            # Handle missing or invalid statuses
            if not status or status.lower() == 'nan':  # Check for empty or "nan" values
                status = 'Healthy'

            # Store the player's status, avoiding duplicates
            players_injuries[player_name] = status
else:
    print("Failed to retrieve injury data.")
    players_injuries = {}  # Initialize as empty dictionary if data retrieval fails


# Create the fetch_injuries function to return the injury data
def fetch_injuries():
    """Returns a dictionary containing player injury statuses."""
    return players_injuries

# Fetch player positions, team defense, player stats, etc. (keep these functions as is)

# Merge the injury status with player data and ensure no duplicates
def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df

# Project PRA for each player against opposing teams
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()
    time.sleep(2)

    print("Fetching injuries...")
    injuries = fetch_injuries()
    time.sleep(2)

    if player_stats_df is None or team_defense_df is None or player_positions_df is None:
        print("Error: One or more dataframes are empty. Returning empty DataFrame.")
        return pd.DataFrame()

    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    print("Projecting PRA...")
    dvp_df["Proj_PTS"] = (dvp_df["PTS"] * (dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean())).round(1)
    dvp_df["Proj_REB"] = (dvp_df["REB"] * (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean())).round(1)
    dvp_df["Proj_AST"] = (dvp_df["AST"] * (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())).round(1)

    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PTS', 'Proj_REB', 'Proj_AST', 'E_USG_PCT']].copy()
    proj_pra_df['Proj_PRA'] = (proj_pra_df['Proj_PTS'] + proj_pra_df['Proj_REB'] + proj_pra_df['Proj_AST']).round(1)

    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    proj_pra_df = merge_with_injuries(proj_pra_df, injuries)

    print("Projection completed successfully!")

    return proj_pra_df if not proj_pra_df.empty else pd.DataFrame()

def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df['Injury_Status'] = proj_pra_df['Injury_Status'].fillna('Healthy')
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df

    print("Projection completed successfully!")

    # Return the DataFrame to be used later
    return proj_pra_df

# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

    for game in todays_games:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame: {home_team_full} vs {away_team_full}")
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | PTS: {row['Proj_PTS']:<5.1f} | REB: {row['Proj_REB']:<5.1f} | AST: {row['Proj_AST']:<5.1f} | PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

# Run the pipeline
if __name__ == "__main__":
    run_pipeline_for_todays_games()

IndexError: single positional indexer is out-of-bounds